## PingOne Overview

PingOne is a cloud-based identity and access management service that provides secure identity solutions for enterprises, enabling seamless authentication and authorization across applications and services.

Key Features:
* **Single Sign-On (SSO)** - Users authenticate once to access multiple applications
* **Multi-Factor Authentication (MFA)** - Enhanced security through additional verification methods
* **Adaptive Authentication** - Risk-based authentication policies based on user behavior and context
* **Universal Directory** - Centralized user management and profile synchronization
* **API Access Management** - OAuth 2.0 and OpenID Connect support for API security

## Amazon Bedrock Gateway Overview

Bedrock AgentCore Gateway provides customers a way to turn their existing APIs and Lambda functions into fully-managed MCP servers without needing to manage infra or hosting. Customers can bring OpenAPI spec or Smithy models for their existing APIs, or add Lambda functions that front their tools. Gateway will provide a uniform Model Context Protocol (MCP) interface across all these tools for agents to consume. Gateway employs a dual authentication model to ensure secure access control for both incoming requests and outbound connections to target resources. The framework consists of two key components: Inbound Auth, which validates and authorizes agents or client applications requesting access to gateway targets, and Outbound Auth, which enables the gateway to securely connect to backend resources on behalf of those authenticated callers. Together, these mechanisms enable agents to securely invoke tools on behalf of authenticated users, supporting both IAM credentials and OAuth-based authentication flows via MCP's Streamable HTTP transport.

More details on Amazon Bedrock AgentCore Gateway can be found at:
- https://github.com/awslabs/amazon-bedrock-agentcore-samples/tree/main/01-tutorials/02-AgentCore-gateway
- https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html

## Learning Objective

PingOne can be used as an identity provider on AgentCore Identity to authorize consuming applications' access to protected Amazon AgentCore Gateway resources. In this notebook we will explore the use of PingOne for inbound authentication with Amazon Bedrock Gateway.

## Tutorial Architecture

Applications authenticate with PingOne using client credentials, then use the resulting JWT to securely invoke Lambda-backed agent tools through the Gateway.
<figure style="width: 70%;">
    <img src="/images/gateway_auth_diagram.png" style="width: 70%; height: auto;" alt="Detailed AWS Cloud Architecture">
</figure>

## Prerequisites

- PingOne environment
- PingOne admin rights to create new resources and applications
- Python 3.10+
- AWS credentials
- AWS IAM rights to create a new AgentCore Gateway and Lambda
- Amazon Bedrock AgentCore SDK
- Strands Agents
- AWS region that supports Bedrock AgentCore. Refer https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-regions.html

## Step 1: Setup PingOne for use with AgentCore Gateway

### 1.1: Create a new Resource

**This will represent AgentCore Gateway, defining the audience and scope that valid tokens must include for access**
1. Login to your PingOne environment
2. Select **Applications**, then **Resources**, and add a resource
3. Configure the resource:
    - For **Resource Name** and **Audience**, enter **agentcore_gateway**
    - For **Scopes**, add **mcp:invoke**

<figure>
    <img src="/images/gateway_resource_config.png">
</figure>

### 1.2: Create a new Application

**This will represent the client, configured to request tokens with the required audience and scope for Agentcore Gateway access**
1. Select **Applications**, then **Applications**, and add an application
2. Select **OIDC Web App** as the **Application Type**
3. Configure the application:
    - For **Application Name**, enter **Agent Orchestrator**
    - For **Token Auth Method**, select **Client Secret Basic**
    - For **Grant Type**, select **Client Credentials**
    - For **Resources**, add the **mcp:invoke** scope on the **agentcore_gateway** resource configured in step 1.1

<figure>
    <img src="/images/gateway_application_config.png">
</figure>

## Step 2: Notebook Setup and AWS Configuration

### 2.1: Install Dependencies for Notebook

In [ ]:
# Install required packages from requirements.txt.
%pip install --force-reinstall -U -r requirements.txt --quiet

### 2.2: Load Environment Variables for Notebook

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

PINGONE_CLIENT_ID = os.environ['GATEWAY_PINGONE_CLIENT_ID']
PINGONE_CLIENT_SECRET = os.environ['GATEWAY_PINGONE_CLIENT_SECRET']
PINGONE_AUDIENCE = os.environ['GATEWAY_PINGONE_AUDIENCE']
PINGONE_SCOPE = os.environ['GATEWAY_PINGONE_SCOPE']
PINGONE_DISCOVERY_URL = os.environ['GATEWAY_PINGONE_DISCOVERY_URL']
PINGONE_TOKEN_URL = os.environ['GATEWAY_PINGONE_TOKEN_URL']

### 2.3: Discover AWS Environment Identity

In [ ]:
import boto3
from boto3.session import Session

# Create an AWS session, picking up credentials and region from environment variables or ~/.aws/credentials.
boto_session = Session()
region = boto_session.region_name

# Create a client for AWS Security Token Service (STS), used for identity operations.
sts = boto3.client('sts')
account_id = sts.get_caller_identity().get('Account')

# Display the target AWS account and region for verification.
print(f'📍 Deploying to AWS Account ID {account_id} in {region}.')

### 2.4: Configure Resource Names

Define names for the AWS resources that will be created. Customize these as needed.

In [ ]:
# Lambda: The function that contains your tool logic (weather_check, directions).
LAMBDA_FUNCTION_NAME = 'pingone-gateway-tools'
LAMBDA_ROLE_NAME = 'pingone-gateway-lambda-role'

# Gateway: The MCP server that exposes your Lambda tools to agents, secured by PingOne JWT auth.
GATEWAY_NAME = 'pingone-gateway'
GATEWAY_ROLE_NAME = 'pingone-gateway-invoke-role'
GATEWAY_DESCRIPTION = 'AgentCore Gateway secured with PingOne JWT authentication'

# Target: The registration that links your Lambda to the Gateway, making its tools available via MCP.
TARGET_NAME = 'LambdaTools'
TARGET_DESCRIPTION = 'Lambda function exposing weather and directions tools'

## Step 3: Deploy a Lambda Function as an Agent Tool

### 3.1: Define Lambda Tool Logic

Write the Lambda handler that AgentCore Gateway will invoke. The tool name is passed via `context` to route requests to the appropriate logic.

In [ ]:
%%writefile lambda_function.py
def lambda_handler(event: dict, context) -> str:
    """
    Lambda handler invoked by AgentCore Gateway when an agent calls a tool.

    Args:
        event: The tool input payload (e.g., {'city': 'Vancouver'})
        context: LambdaContext with runtime metadata including client_context from Gateway

    Returns:
        The tool's response string
    """
    # AgentCore Gateway passes the tool name via client context
    qualified_tool_name = context.client_context.custom['bedrockAgentCoreToolName']
    tool_name = qualified_tool_name.split('___')[1]
    print(f'Tool: {tool_name}')

    # Extract the 'city' parameter from the tool input.
    city = event.get('city')
    print(f'City: {city}')

    # Route to the appropriate tool logic based on the tool name.
    if tool_name == 'weather_check':
        return f'Weather in {city} is bright and sunny!'
    elif tool_name == 'directions':
        return f'Take I5 south all the way to {city} downtown'
    else:
        raise ValueError(f'Unknown tool: {tool_name}')

### 3.2: Package Lambda for Deployment

Bundle the function code into a ZIP archive, which is the format Lambda expects for deployment.

In [ ]:
import zipfile

# Zip lambda_function.py into lambda_function.zip
with zipfile.ZipFile('lambda_function.zip', 'w') as zip_file:
    zip_file.write('lambda_function.py', 'lambda_function.py')

# Read lambda_function.zip into zip_content (raw bytes)
with open('lambda_function.zip', 'rb') as zip_file:
    zip_content = zip_file.read()

### 3.3: Create Lambda Execution Role

Define an IAM role that grants the Lambda function permission to write logs to CloudWatch.

In [ ]:
import time

# Create an IAM client to interact with the AWS IAM service.
iam_client = boto3.client('iam', region_name=region)

# Trust policy: allows the AWS Lambda service to assume this role.
lambda_trust_policy = '''{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": { "Service": "lambda.amazonaws.com" },
            "Action": "sts:AssumeRole"
        }
    ]
}'''

# Permissions policy: grants the role access to write CloudWatch logs.
lambda_permission_policy = '''{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "arn:aws:logs:*:*:*"
        }
    ]
}'''

# Create the IAM role with the trust policy, or use existing if already created.
try:
    response = iam_client.create_role(
        RoleName=LAMBDA_ROLE_NAME,
        AssumeRolePolicyDocument=lambda_trust_policy
    )
    lambda_role_arn = response['Role']['Arn']
    print(f'Created new role: {lambda_role_arn}')
    # New IAM roles take time to propagate across AWS.
    print('Waiting 10s for new role to propagate...')
    time.sleep(10)
except iam_client.exceptions.EntityAlreadyExistsException:
    response = iam_client.get_role(RoleName=LAMBDA_ROLE_NAME)
    lambda_role_arn = response['Role']['Arn']
    print(f'Using existing role: {lambda_role_arn}')

# Attach the permissions policy to the role, granting the role CloudWatch Logs access.
iam_client.put_role_policy(
    PolicyDocument=lambda_permission_policy,
    PolicyName='lambda-logging-policy',
    RoleName=LAMBDA_ROLE_NAME
)

### 3.4: Deploy Lambda Function

Create the Lambda function in AWS using the packaged code and execution role defined above.

In [ ]:
# Create a Lambda client to interact with the AWS Lambda service.
lambda_client = boto3.client('lambda', region_name=region)

# Create the Lambda function, or update it if it already exists.
try:
    response = lambda_client.create_function(
        FunctionName=LAMBDA_FUNCTION_NAME,
        Runtime='python3.12',
        Role=lambda_role_arn,
        Handler='lambda_function.lambda_handler',
        Code={'ZipFile': zip_content},
    )
    print(f'Successfully created function: {LAMBDA_FUNCTION_NAME}')
except lambda_client.exceptions.ResourceConflictException:
    response = lambda_client.update_function_code(
        FunctionName=LAMBDA_FUNCTION_NAME,
        ZipFile=zip_content,
    )
    print(f'Function already exists. Updated code instead.')

# Retrieve the function ARN for use when configuring the Gateway target.
response = lambda_client.get_function(FunctionName=LAMBDA_FUNCTION_NAME)
lambda_arn = response['Configuration']['FunctionArn']
print(f'Target Lambda ARN: {lambda_arn}')

## Step 4: Configure AgentCore Gateway

### 4.1: Create AgentCore Gateway IAM Role

Define an IAM role that allows AgentCore Gateway to invoke your Lambda function.

In [ ]:
# Trust policy: allows the AWS AgentCore Gateway service to assume this role.
gateway_trust_policy = '''{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "bedrock-agentcore.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}
'''

# Permissions policy: grants the role access to invoke your Lambda function.
gateway_permission_policy = '''{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Action": [
                "lambda:InvokeFunction"
            ],
            "Resource": [
                "%s"
            ],
            "Effect": "Allow",
            "Sid": "InvokeFunction"
        }
    ]
}
''' % lambda_arn

# Create the IAM role with the trust policy, or use existing if already created.
try:
    response = iam_client.create_role(
        RoleName=GATEWAY_ROLE_NAME,
        AssumeRolePolicyDocument=gateway_trust_policy
    )
    gateway_role_arn = response['Role']['Arn']
except iam_client.exceptions.EntityAlreadyExistsException:
    response = iam_client.get_role(RoleName=GATEWAY_ROLE_NAME)
    gateway_role_arn = response['Role']['Arn']

# Attach the permissions policy to the role, granting Lambda invocation to your AgentCore Gateway role.
iam_client.put_role_policy(
    RoleName=GATEWAY_ROLE_NAME,
    PolicyName='lambda-invoke-policy',
    PolicyDocument=gateway_permission_policy
)
print(f'Gateway Role ARN: {gateway_role_arn}')

### 4.2: Create the AgentCore Gateway

Set up the Gateway with Custom JWT authorization so it validates incoming requests against your PingOne environment.

In [ ]:
# Create a gateway client to interact with the AWS AgentCore Gateway service.
gateway_client = boto3.client('bedrock-agentcore-control', region_name=region)

# Check if a gateway with this name already exists.
existing_gateways = gateway_client.list_gateways().get('items', [])
existing_gateway = next((g for g in existing_gateways if g['name'] == GATEWAY_NAME), None)

if existing_gateway:
    # Gateway exists - retrieve its ID and URL.
    gateway_id = existing_gateway['gatewayId']
    detail_response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
    gateway_url = detail_response['gatewayUrl']
    print(f'Using existing gateway: {gateway_id}')
else:
    # Configure JWT auth to validate tokens from PingOne.
    # Gateway will verify the audience, issuer (via discovery URL), and required scopes.
    auth_config = {
        'customJWTAuthorizer': {
            'allowedAudience': [PINGONE_AUDIENCE],
            'discoveryUrl': PINGONE_DISCOVERY_URL,
            'allowedScopes': [PINGONE_SCOPE]
        }
    }
    # Create the gateway with MCP protocol and custom JWT authorization.
    create_response = gateway_client.create_gateway(
        name=GATEWAY_NAME,
        roleArn=gateway_role_arn,
        protocolType='MCP',
        authorizerType='CUSTOM_JWT',
        authorizerConfiguration=auth_config,
        description=GATEWAY_DESCRIPTION,
    )
    gateway_id = create_response['gatewayId']
    gateway_url = create_response['gatewayUrl']
    print(f'Created new gateway: {gateway_id}')

print(f'Gateway URL: {gateway_url}')

## Step 5: Register Lambda as a Gateway Target

### 5.1: Define Tool Schemas

Specify the tools your Lambda exposes, including their names, descriptions, and input parameters.

In [ ]:
tool_schemas = [
    {
        'name': 'weather_check',
        'description': 'Check the weather for a given city',
        'inputSchema': {
            'type': 'object',
            'properties': {
                'city': {
                    'type': 'string',
                    'description': 'The city you want to get weather for'
                }
            },
            'required': ['city']
        }
    },
    {
        'name': 'directions',
        'description': 'Get directions to a city',
        'inputSchema': {
            'type': 'object',
            'properties': {
                'city': {
                    'type': 'string',
                    'description': 'The city you want to get directions to'
                }
            },
            'required': ['city']
        }
    }
]

### 5.2: Register Lambda as Gateway Target

Link your Lambda function to the Gateway, officially registering it as a "Target" that can be reached via the Gateway's MCP interface.

In [ ]:
# Check if a target with this name already exists on the Gateway.
existing_targets = gateway_client.list_gateway_targets(
    gatewayIdentifier=gateway_id
).get('items', [])
existing_target = next((t for t in existing_targets if t['name'] == TARGET_NAME), None)

if existing_target:
    # Target exists - retrieve its ID.
    target_id = existing_target['targetId']
    print(f'✅ Target {TARGET_NAME} already exists. Target ID: {target_id}')
else:
    # Configure the target: point to Lambda ARN and attach the tool schema.
    lambda_target_config = {
        'mcp': {
            'lambda': {
                'lambdaArn': lambda_arn,
                'toolSchema': {'inlinePayload': tool_schemas},
            }
        }
    }
    # Register the Lambda as a Gateway target.
    # Uses the Gateway's IAM role (created earlier) to invoke the Lambda.
    create_target_response = gateway_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name=TARGET_NAME,
        description=TARGET_DESCRIPTION,
        targetConfiguration=lambda_target_config,
        credentialProviderConfigurations=[{'credentialProviderType': 'GATEWAY_IAM_ROLE'}],
    )
    target_id = create_target_response['targetId']
    print(f'🚀 Successfully created new target. Target ID: {target_id}')

print(f'Target is ready for use in Gateway: {gateway_id}')

## Step 6: Invoke Gateway Tools from an Agent

### 6.1: Fetch an Access Token

Request a JWT from PingOne using client credentials.

In [ ]:
import requests

def fetch_access_token() -> str:
    """Fetch an access token from PingOne using client credentials."""
    response = requests.post(
        PINGONE_TOKEN_URL,
        data={
            'grant_type': 'client_credentials',
            'scope': PINGONE_SCOPE
        },
        auth=(PINGONE_CLIENT_ID, PINGONE_CLIENT_SECRET),
        headers={'Content-Type': 'application/x-www-form-urlencoded'}
    )
    if response.status_code != 200:
        raise RuntimeError(f'Failed to fetch token: {response.status_code} {response.text}')
    return response.json()['access_token']

access_token = fetch_access_token()

### 6.2: Inspect the Token

Decode the JWT to verify the issuer, audience, and scope match your PingOne configuration.

In [ ]:
import base64
import json

def decode_jwt_payload(token: str) -> dict:
    """Decode the payload from a JWT token (without verification)."""
    parts = token.split('.')
    payload = json.loads(base64.b64decode(parts[1] + '==').decode('utf-8'))
    return payload

payload = decode_jwt_payload(access_token)
print(f"Issuer: {payload.get('iss')}")
print(f"Audience: {payload.get('aud')}")
print(f"Scope: {payload.get('scope')}")
print(f"Expires: {payload.get('exp')}")

### 6.3: List Available Tools (Raw HTTP)

Demonstrate the raw MCP protocol call to the Gateway's `tools/list` endpoint. In practice, use the MCP client (shown in 6.4).

In [ ]:
def list_tools(gateway_url, access_token):
    response = requests.post(
        gateway_url,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {access_token}'
        },
        json={
            'jsonrpc': '2.0',
            'id': 'list-tools-request',
            'method': 'tools/list'
        }
    )
    return response.json()

tools = list_tools(gateway_url, access_token)
print(json.dumps(tools, indent=2))

### 6.4: Create an MCP Client

Connect to the Gateway and retrieve the list of available tools.

In [ ]:
import httpx
from mcp.client.streamable_http import streamable_http_client
from strands.tools.mcp import MCPClient

# Create an HTTP client with the PingOne access token.
http_client = httpx.AsyncClient(
    headers={'Authorization': f'Bearer {access_token}'},
    timeout=httpx.Timeout(30.0, read=300.0)
)

# Set up MCP client using the authenticated HTTP client.
mcp_client = MCPClient(
    lambda: streamable_http_client(
        url=gateway_url,
        http_client=http_client,
    )
)

mcp_client.start()
mcp_client.list_tools_sync()

### 6.5: Invoke Tools via the Agent

Create a Strands agent with the MCP tools and test tool invocation.

**Note:** The agent doesn't need the token directly—it invokes tools through the MCP client, which handles authentication.

In [ ]:
from strands import Agent
agent = Agent(tools=mcp_client.list_tools_sync())

In [ ]:
agent('What is the weather in Vancouver?')

In [ ]:
agent('Give me directions to Vancouver?')

## Conclusion and Cleanup

### Resource Cleanup

Clean up the resources created in this tutorial:

In [ ]:
# Stop the MCP client before cleanup.
mcp_client.stop(None, None, None)

# 1. Delete Lambda target on your Gateway.
print('🗑️ Deleting gateway target...')
try:
    gateway_client.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=target_id)
    print(f'✅ Target {target_id} deleted.')
    # Wait for detachment only if a deletion actually happened.
    print('⏳ Waiting for detachment...')
    time.sleep(10)
except gateway_client.exceptions.ResourceNotFoundException:
    print(f'ℹ️ Target {target_id} already gone. Skipping...')

# 2. Delete Gateway.
print('🗑️ Deleting gateway...')
try:
    gateway_client.delete_gateway(gatewayIdentifier=gateway_id)
    print(f'✅ Gateway {gateway_id} deleted.')
except gateway_client.exceptions.ResourceNotFoundException:
    print(f'ℹ️ Gateway {gateway_id} not found. Skipping...')

# 3. Delete Lambda function.
print('🗑️ Deleting Lambda function...')
try:
    lambda_client.delete_function(FunctionName=LAMBDA_FUNCTION_NAME)
    print(f'✅ Lambda {LAMBDA_FUNCTION_NAME} deleted.')
except lambda_client.exceptions.ResourceNotFoundException:
    print(f'ℹ️ Lambda {LAMBDA_FUNCTION_NAME} already deleted.')

# 4. Delete created IAM roles.
print('🗑️ Deleting IAM roles...')
for role_name in [LAMBDA_ROLE_NAME, GATEWAY_ROLE_NAME]:
    try:
        inline = iam_client.list_role_policies(RoleName=role_name)
        for policy_name in inline['PolicyNames']:
            iam_client.delete_role_policy(RoleName=role_name, PolicyName=policy_name)
        iam_client.delete_role(RoleName=role_name)
        print(f'✅ Role {role_name} deleted.')
    except iam_client.exceptions.NoSuchEntityException:
        print(f'ℹ️ Role {role_name} already deleted.')

print('✨ Cleanup finished.')

### Conclusion

This notebook demonstrated how to:
1. **Configure PingOne** - Create a resource and application for the `mcp:invoke` scope
2. **Deploy a Lambda Tool** - Package and deploy a Lambda function with tool logic
3. **Provision AgentCore Gateway** - Create a gateway with Custom JWT authorization to validate PingOne tokens
4. **Register the Lambda as a Target** - Link the Lambda to the Gateway, exposing its tools via MCP
5. **Invoke Tools from an Agent** - Use a Strands agent to call the secured tools through the Gateway

### Key Learnings

- **Inbound Auth with Custom JWT:** The Gateway validates PingOne JWTs automatically—checking issuer, audience, and scopes before allowing tool invocation.
- **MCP as a Uniform Interface:** The Gateway exposes Lambda functions (and APIs) through a standard MCP interface, abstracting the underlying implementation.
- **Separation of Concerns:** Authentication is handled at the Gateway layer, so your Lambda code stays focused on business logic.

### Next Steps

- **Outbound Auth:** Configure the Gateway to authenticate with backend APIs on behalf of users using OAuth flows.
- **OpenAPI Targets:** Register REST APIs via OpenAPI spec instead of Lambda functions.
- **Multiple Scopes:** Define granular scopes (e.g., `mcp:read`, `mcp:write`) to control which tools different clients can access.